In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.prompts import ChatPromptTemplate

C:\Users\Ritika Khandelwal\AppData\Local\Temp\ipykernel_25064\837613102.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [13]:

pdf1='sample-data/documents/handwritten/HANDWRITTEN_002.pdf'

data = PyMuPDFLoader(pdf1).load()

In [14]:
pdf2 = 'sample-data/documents/mutation/MUTATION_001.pdf'
data2= PyMuPDFLoader(pdf2).load()


In [15]:
print(data2[0].page_content)

नामांतरण पंɣजिका / आदेश - पृष्ठ १
LAND MUTATION REGISTER / ORDER ENTRY — PAGE 1 OF 2
[ IntelliLand AI Validation Testbed — Reference: MUTATION_001 ]
नामांतरण संख्या
Mutation No.
MUT-2026-88319
आवेदन ɟतɡथि
Mutation Date
22-Jul-2026
ɣजिला
District
सूरजपुर (काल्पɟनिक) / Surajpur
(Fictional)
तहसील
Tehsil
देवगढ़ / Devgarh
ग्राम
Village
गोपालपुर / Gopalpur
नामांतरण प्रकार
Mutation Type
Inheritance / वरासत (उत्तराɠधिकार)
भाग १: अंतरणकतार्ता एवं अंतɝरती ɟववरण (Part 1: Transferor & Transferee Details) 
पूवर्ता भू-स्वामी (मूल खातेदार)
Previous Owner / Transferor
नवीन भू-स्वामी (आवेदक)
New Owner / Transferee
संबंध / अंतरण कारण
Relation / Reason
स्व. रामप्रसाद शमार्ता
Late Ramprasad Sharma
सुरेश कुमार
Suresh Kumar
उत्तराɠधिकार (पुत्र) / ɟवɠधिक वाɝरस
Inheritance (Son) / Legal Heir
[ Continued on Page 2 / पृष्ठ २ पर जारी... ]
SYNTHETIC PROTOTYPE — NOT A GOVERNMENT
RECORD
कृɟत्रिम प्रारूप — यह शासकीय अɢभलेख नहीं है


In [14]:
print(data[0].page_content)

पुरातन भू-अɢभलेख रɣजिस्टर (प्रɟविɟष्टि वि;र्ष १९७९)
LEGACY HANDWRITTEN LAND RECORD — CHALLENGE SPECIMEN
[ IntelliLand AI Stress Test Suite — Specimen ID: HANDWRITTEN_002 ]
अɢभलेख संदभर्ष संख्या
Record Ref No.
प्रɟविɟष्टि ɟतɡथि
Record Date
ɣजिला
District
 / Rampur
(Fictional)
तहसील
Tehsil
 / Anandnagar
ग्राम
Village
 / Chandanpur
खाता संख्या
Khata No.
हस्तɡलɤखत प्रɟविɟष्टि एविं क्षेत्रफल ɟविविरण (Handwritten Entries & Land Metrics) 
खातेदार का नाम
Owner Name
ɟपता का नाम
Father Name
खसरा सं.
Khasra No.
क्षेत्रफल (हेक्टेयर)
Area (Hectares)
(Hariprasad Verma)
(Late Ramdeen Verma)
अभ्युɜक्त
Remarks
(Legacy record updated. Revenue receipt no. 741 logged.)
[ SIGNATURE PLACEHOLDER ]
Revenue Officer / राजस्व कानूनगो (1979) 
DISCLAIMER: Fictional degraded handwritten test document created for AI OCR stress testing and human validation review in IntelliLand. Page 1
of 1. 
LEGACY-1979/4102
10-Nov-1979
रामपुर (काल्पɟनक)
आनंदनगर
चंदनपुर
142/A
हɝरप्रसाद विमार्ष
स्वि. रामदीन विमार्ष
589/2
 0.8600
Ha 


In [8]:
# This is a google cloud llm instance
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    vertexai=True,
    project="ai-api-2335",  # Replace with your GCP project ID
    location="us-central1",
    temperature=0
)

In [5]:
# app/schemas/land_record.py
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional

class OwnerDetail(BaseModel):
    name: str
    relation_type: Optional[str] = None  # e.g., "S/o", "W/o"
    relative_name: Optional[str] = None
    share_fraction: Optional[str] = "1/1"

class ParcelRecord(BaseModel):
    khasra_number: str
    khata_number: Optional[str] = None
    area_raw: str
    area_sq_meters: float
    land_type: str = Field(description="Agricultural, Residential, etc.")
    owners: List[OwnerDetail]
    confidence_score: float

class LandRecordResponse(BaseModel):
    state: str
    district: str
    tehsil: str
    owner: OwnerDetail
    village: str
    records: List[ParcelRecord]
    requires_human_review: bool

In [25]:


SYSTEM_PROMPT = """You are an expert Indian land records (Bhulekh / Jamabandi / RoR) extraction engine.
Your task is to parse unstructured OCR or typed land record text and extract structured entities matching the schema.

Guidelines for extraction:
1. **Administrative Divisions**:
   - Accurately identify `state`, `district`, `tehsil`, and `village`.
   - Preserve standardized names. If both vernacular (Hindi) and English names appear, output the clean English name or standard transliteration.

2. **Parcel Records (`records`)**:
   - `khasra_number`: Extract the survey/khasra number (e.g., "741/3", "120/1").
   - `khata_number`: Extract the ledger/account number (e.g., "96/B", "102").
   - `area_raw`: Record the exact area string as written in the text (e.g., "2 बीघा 4 बिस्वा", "1.8500 Ha", "0.4050 Hectare").
   - `area_sq_meters`: Convert the extracted area to square meters ($m^2$) as a float.
     * 1 Hectare = 10,000 m²
     * 1 Acre = 4,046.86 m²
     * Standard Bigha conversions (unless regionally qualified): 1 Bigha ≈ 2,500 m² (or standard 2,529.29 m²), 1 Biswa ≈ 1/20 Bigha.
     * If multiple units are provided (e.g., Bigha and Hectares), prioritize the Hectare measurement for the metric conversion.
   - `land_type`: Standardize the classification (e.g., "Agricultural", "Residential", "Commercial", "Canal Irrigated Agricultural").
   - `confidence_score`: Estimate overall extraction accuracy for this specific parcel between 0.0 and 1.0 based on OCR legibility, missing data, or conversion ambiguity.

3. **Owner Details (`owners`)**:
   - `name`: Clean owner name without relational prefixes.
   - `relation_type`: Normalize relations to standard tokens: "S/o" (Son of), "D/o" (Daughter of), "W/o" (Wife of), "C/o", or null if not indicated.
   - `relative_name`: Name of the parent/spouse.
   - `share_fraction`: Normalized ownership fraction (e.g., "1/1", "1/2", "1/4"). Defaults to "1/1" if unspecified.

4. **Human Review Flag (`requires_human_review`)**:
   - Set to `true` if:
     * Khasra or Khata numbers are smudged, ambiguous, or missing.
     * Area conversion contains unresolvable regional units.
     * Any parcel's `confidence_score` is below 0.70.
     * Remarks indicate disputes, court stays, or active litigation.
   - Otherwise, set to `false`.

Do not extrapolate or hallucinate details not grounded in the source text.
"""
all_data = '/n'.join([d.page_content for d in data])
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        (
            "human",
            "Extract the structured land record information from the following text:\n\n{raw_text}",
        ),
    ]
)
res_llm = prompt  | llm.with_structured_output(LandRecordResponse)
res=res_llm.invoke({'raw_text': all_data})
print(res)


state='Fictional State' district='Rampur' tehsil='Anandnagar' owner=OwnerDetail(name='Hariprasad Verma', relation_type='S/o', relative_name='Ramdeen Verma', share_fraction='1/1') village='Chandanpur' records=[ParcelRecord(khasra_number='589/2', khata_number='142/A', area_raw='0.8600 Ha 0.9000 Ha (1 बीघा 2 बिस्वा)', area_sq_meters=8600.0, land_type='Agricultural', owners=[OwnerDetail(name='Hariprasad Verma', relation_type='S/o', relative_name='Ramdeen Verma', share_fraction='1/1')], confidence_score=0.85)] requires_human_review=True


In [26]:
json_data = res.model_dump_json(indent=2)
print(json_data)

{
  "state": "Fictional State",
  "district": "Rampur",
  "tehsil": "Anandnagar",
  "owner": {
    "name": "Hariprasad Verma",
    "relation_type": "S/o",
    "relative_name": "Ramdeen Verma",
    "share_fraction": "1/1"
  },
  "village": "Chandanpur",
  "records": [
    {
      "khasra_number": "589/2",
      "khata_number": "142/A",
      "area_raw": "0.8600 Ha 0.9000 Ha (1 बीघा 2 बिस्वा)",
      "area_sq_meters": 8600.0,
      "land_type": "Agricultural",
      "owners": [
        {
          "name": "Hariprasad Verma",
          "relation_type": "S/o",
          "relative_name": "Ramdeen Verma",
          "share_fraction": "1/1"
        }
      ],
      "confidence_score": 0.85
    }
  ],
  "requires_human_review": true
}


In [19]:
res=LandRecordResponse(state='null', district='Rampur', tehsil='Anandnagar', village='Chandanpur', records=[ParcelRecord(khasra_number='589/2', khata_number='142/A', area_raw='0.8600 Ha 0.9000 Ha (1 बीघा 2 ɟबस्विा)', area_sq_meters=8600.0, land_type='Agricultural', owners=[OwnerDetail(name='Hariprasad Verma', relation_type='S/o', relative_name='Ramdeen Verma', share_fraction='1/1')], confidence_score=0.9)], requires_human_review=True)

In [22]:
for r in res:
    print(r)

('state', 'null')
('district', 'Rampur')
('tehsil', 'Anandnagar')
('village', 'Chandanpur')
('records', [ParcelRecord(khasra_number='589/2', khata_number='142/A', area_raw='0.8600 Ha 0.9000 Ha (1 बीघा 2 ɟबस्विा)', area_sq_meters=8600.0, land_type='Agricultural', owners=[OwnerDetail(name='Hariprasad Verma', relation_type='S/o', relative_name='Ramdeen Verma', share_fraction='1/1')], confidence_score=0.9)])
('requires_human_review', True)


In [4]:
document='ai-service/REGISTRATION_001_POOR_QUALITY_page-0001.jpg'

In [ ]:
from google import genai
from google.genai.types import Part

client = genai.Client(
    vertexai=True,
    project="ai-api-2335",
    location="global"
)

with open("ai-service/REGISTRATION_001_POOR_QUALITY_page-0001.jpg", "rb") as f:
    image_bytes = f.read()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        "Extract all the text from this document. Preserve the original language.",
        Part.from_bytes(
            data=image_bytes,
            mime_type="image/jpeg"
        )
    ]
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Here is the extracted text from the document, preserving the original language:

SUB-REGISTRAR OFFICE — PROPERTY SALE DEED
उप-पंजीयक कार्यालय - संपत्ति विक्रय विलेख
IntelliLane AI Validation Dataset - eDocument SUB-REGISTRATION

Registration No. /
पंजीयन सं.
REG-2026-004812

Registration Date /
तिथि
14-Aug-2026

District / जिला
Surajpur (जिले का नाम)

Tehsil / तहसील
Devgarh (तहसील)

Village / ग्राम
Gopalpur (ग्राम)

Stamp Duty Paid /
मुद्रांक शुल्क
₹ 315,000

1. PARTIES TO THE SALE DEED (पक्षकारों का विवरण)
Seller / Transferor (विक्रेता)
Name: Ramesh Chandra Verma (रमेश चंद्र वर्मा)
Father's Name: Late Badi Prasad Verma

Buyer / Transferee: (क्रेता)
Name: Anil Kumar Srivastava (अनिल कुमार श्रीवास्तव)
Father's Name: Harish Chandra Srivastava

2. PROPERTY & TRANSACTION DETAILS (संपत्ति व अंतरण विवरण)
Survey Number:
BV-402

Khasra Number:
781/9

Land Area:
2.8500 Ha (4.571 Acres)

Consideration Value:
₹ 4,500,000 (INR Forty Five Lakhs Only)

Property Description:
Agricultural land/plot si

In [ ]:
llm.invoke(contents=[
        "Extract all the text from this document. Preserve the original language.",
        Part.from_bytes(
            data=image_bytes,
            mime_type="image/jpeg"
        )
    ])

In [ ]:
## Using langchain
import base64

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage



with open(document, "rb") as f:
    image_data = base64.b64encode(f.read()).decode("utf-8")

message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Extract all text from this document. Preserve the original language."
        },
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{image_data}"
            }
        }
    ]
)
res_llm =  llm.with_structured_output(LandRecordResponse)
response = res_llm.invoke([message])



AttributeError: 'LandRecordResponse' object has no attribute 'content'

In [11]:
print(response.model_dump_json(indent=2))

{
  "state": "",
  "district": "Suraipur",
  "tehsil": "Devgarh",
  "owner": {
    "name": "Anl Srivastava",
    "relation_type": "Father",
    "relative_name": "Harish Chandra",
    "share_fraction": null
  },
  "village": "Gopalpur",
  "records": [
    {
      "khasra_number": "781/A",
      "khata_number": null,
      "area_raw": "2.8500 Ha (4.571 ACRES)",
      "area_sq_meters": 28500.0,
      "land_type": "Agricultural",
      "owners": [
        {
          "name": "Anl Srivastava",
          "relation_type": "Father",
          "relative_name": "Harish Chandra",
          "share_fraction": null
        }
      ],
      "confidence_score": 1.0
    }
  ],
  "requires_human_review": true
}
